In [1]:
import pandas as pd
import numpy as np

print("Libraries loaded successfully!")

Libraries loaded successfully!


In [2]:
# Load AID_720637 - skip the 4 metadata rows (RESULT_TYPE, RESULT_DESCR, RESULT_UNIT, RESULT_IS_ACTIVE_CONCENTRATION)
mmp_raw = pd.read_csv(
    '/Users/yamini/Desktop/projects/qsar casepase/qsar_prediction/Rebuttal/summary datasets/AID_720637_datatable (1).csv',
    skiprows=[1, 2, 3, 4],
    low_memory=False)

print("AID_720637 - MMP + viability summary, HepG2")
print(f"Loaded shape: {mmp_raw.shape}")

# Remove rows with missing CID
mmp_raw['PUBCHEM_CID'] = pd.to_numeric(mmp_raw['PUBCHEM_CID'], errors='coerce')
print(f"Rows with missing CID: {mmp_raw['PUBCHEM_CID'].isna().sum()}")
mmp_raw = mmp_raw.dropna(subset=['PUBCHEM_CID'])
mmp_raw['PUBCHEM_CID'] = mmp_raw['PUBCHEM_CID'].astype(int)
mmp_raw['PUBCHEM_ACTIVITY_SCORE'] = pd.to_numeric(mmp_raw['PUBCHEM_ACTIVITY_SCORE'], errors='coerce')

print(f"After cleanup: {mmp_raw.shape}")
print(f"Unique CIDs: {mmp_raw['PUBCHEM_CID'].nunique()}")

AID_720637 - MMP + viability summary, HepG2
Loaded shape: (10486, 22)
Rows with missing CID: 149
After cleanup: (10337, 22)
Unique CIDs: 8099


In [3]:
# Merge duplicates per CID:
# 1. Average PUBCHEM_ACTIVITY_SCORE across ALL measurements for that CID,
#    regardless of what outcome each measurement reported.
# 2. Classify the outcome from the averaged score itself, using this assay's
#    own score ranges: Inactive is always 0, Inconclusive is 5-30, Active is
#    40-100 (with a natural, unused gap between 30 and 40). No 'Conflict'
#    category -- every compound gets a real outcome from its averaged score.

def outcome_from_score(score):
    if score == 0:
        return 'Inactive'
    elif score < 40:
        return 'Inconclusive'
    else:
        return 'Active'

agg_dict = {col: 'first' for col in mmp_raw.columns
            if col not in ['PUBCHEM_CID', 'PUBCHEM_ACTIVITY_SCORE', 'PUBCHEM_ACTIVITY_OUTCOME']}
agg_dict['PUBCHEM_ACTIVITY_SCORE'] = 'mean'

mmp_clean = mmp_raw.groupby('PUBCHEM_CID', as_index=False).agg(agg_dict)
mmp_clean['PUBCHEM_ACTIVITY_OUTCOME'] = mmp_clean['PUBCHEM_ACTIVITY_SCORE'].apply(outcome_from_score)

print(f"After dedup: {mmp_raw.shape} -> {mmp_clean.shape}")
print(f"Removed {len(mmp_raw) - len(mmp_clean)} duplicate rows")
print()
print(mmp_clean['PUBCHEM_ACTIVITY_OUTCOME'].value_counts())

After dedup: (10337, 22) -> (8099, 22)
Removed 2238 duplicate rows

PUBCHEM_ACTIVITY_OUTCOME
Inactive        5080
Inconclusive    2162
Active           857
Name: count, dtype: int64


In [4]:
# Verify on a CID whose replicates actually disagreed on outcome
n_outcomes_per_cid = mmp_raw.groupby('PUBCHEM_CID')['PUBCHEM_ACTIVITY_OUTCOME'].nunique()
disagreed_cids = n_outcomes_per_cid[n_outcomes_per_cid > 1].index
example_cid = 727

print(f"Verification - CID {example_cid} (replicates disagreed on outcome):")
print("\nBefore dedup (all replicate rows):")
display(mmp_raw[mmp_raw['PUBCHEM_CID'] == example_cid][['PUBCHEM_CID', 'PUBCHEM_ACTIVITY_OUTCOME', 'PUBCHEM_ACTIVITY_SCORE']])

print("\nAfter dedup (one merged row -- score averaged, outcome from the averaged score):")
display(mmp_clean[mmp_clean['PUBCHEM_CID'] == example_cid][['PUBCHEM_CID', 'PUBCHEM_ACTIVITY_OUTCOME', 'PUBCHEM_ACTIVITY_SCORE']])

print(f"\nTotal CIDs whose replicates disagreed on outcome: {len(disagreed_cids)}")
print("All of these now get a real Active/Inactive/Inconclusive label from their")
print("averaged score -- no 'Conflict' category, nothing dropped.")

Verification - CID 727 (replicates disagreed on outcome):

Before dedup (all replicate rows):


,PUBCHEM_CID,PUBCHEM_ACTIVITY_OUTCOME,PUBCHEM_ACTIVITY_SCORE
1293,727,Inactive,0
4320,727,Active,41
5421,727,Inconclusive,25
5713,727,Inconclusive,5
5934,727,Inactive,0
7387,727,Inconclusive,30
7716,727,Inconclusive,5
9688,727,Inconclusive,25



After dedup (one merged row -- score averaged, outcome from the averaged score):


,PUBCHEM_CID,PUBCHEM_ACTIVITY_OUTCOME,PUBCHEM_ACTIVITY_SCORE
73,727,Inconclusive,16.375



Total CIDs whose replicates disagreed on outcome: 501
All of these now get a real Active/Inactive/Inconclusive label from their
averaged score -- no 'Conflict' category, nothing dropped.


In [5]:
# Save
mmp_clean.to_csv('AID_720637_cleaned.csv', index=False)
print(f"Saved {len(mmp_clean)} compounds to AID_720637_cleaned.csv")

Saved 8099 compounds to AID_720637_cleaned.csv
